In [ ]:
!pip install vllm bitsandbytes pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 144.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.

In [ ]:
import os, subprocess

result = subprocess.run(["find", "/usr", "-name", "libcuda.so*"], capture_output=True, text=True)
print(result.stdout)

/usr/local/cuda-12.8/compat/libcuda.so.570.124.06
/usr/local/cuda-12.8/compat/libcuda.so.1
/usr/local/cuda-12.8/compat/libcuda.so
/usr/local/cuda-12.8/targets/x86_64-linux/lib/stubs/libcuda.so
/usr/lib64-nvidia/libcuda.so.1
/usr/lib64-nvidia/libcuda.so.580.82.07
/usr/lib64-nvidia/libcuda.so



In [ ]:
os.environ["LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# clear the flashinfer JIT cache so it recompiles with the correct paths
!rm -rf /root/.cache/flashinfer/

In [ ]:
server = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--dtype", "float16",
    "--tensor-parallel-size", "1",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.90",
    "--enable-chunked-prefill",
    "--max-num-batched-tokens", "512",
    "--cudagraph-capture-sizes", "1", "2", "4", "8", "16",
    "--port", "8000",
],
    env={**os.environ},
    stdout=open("/content/vllm.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print("Server PID:", server.pid)
with open("/content/vllm.pid", "w") as f:
    f.write(str(server.pid))

Server PID: 2891


In [ ]:
import time, subprocess

print("Waiting for server...")
for _ in range(120):
    time.sleep(60)
    result = subprocess.run(["grep", "-c", "Application startup complete", "/content/vllm.log"],
                            capture_output=True, text=True)
    if result.stdout.strip() == "1":
        print("*** SERVER READY ***")
        break
    # show last line of log so you can see progress
    tail = subprocess.run(["tail", "-1", "/content/vllm.log"], capture_output=True, text=True)
    print(tail.stdout.strip())
else:
    print("Timed out — check /content/vllm.log")

Waiting for server...
*** SERVER READY ***


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3AVBps8CZBIFHhuOziZytnu1rPF_6kjruq6p7y1vkWYcUwywE")
tunnel = ngrok.connect(8000)
print("PUBLIC URL:", tunnel.public_url)

PUBLIC URL: https://rufflike-heteromorphic-lacie.ngrok-free.dev


In [ ]:
ngrok.kill()